In [17]:
from collections import namedtuple

ModelConfig = namedtuple('ModelConfig', ['vocab_size', 'context_length', 'n_layers', 'd_model', 'n_heads', 'd_ff'])

In [32]:
# Get the total # of trainable model params 
def model_params(config):
    # Embedding is a vocab_size * d_model matrix lookup
    emb = config.vocab_size * config.d_model
    # RmsNorm has a (d_model,) 1D tensor as weights
    norm = config.d_model
    # (Multi-head) Self attention has 4 linear layers, each of size (d_model, d_model), for QKV and Ouptut projection O.
    attention = 4 * config.d_model * config.d_model
    # Feed forward / SwigLU has 3 linear layers / matrix multiplication, each of size (d_model, d_ff)
    swiglu = 3 * config.d_model * config.d_ff
    # Total # of params for one transformer blocker
    transformer_block = 2 * norm + attention + swiglu
    # Output linear layer is of size (d_model, vocab_size)
    out_linear = config.d_model * config.vocab_size
    print(f"Attention has {attention * config.n_layers:_} params, Feed forward {swiglu * config.n_layers:_} params, out linear {out_linear:_} params, emb {emb:_} params")
    return emb + config.n_layers * transformer_block + norm + out_linear

# Get total # of flops to run forward pass on a single batch of input with (context_length, d_model) shape
def flops(config):
    # (Multi-head) Self attention has 4 linear layers, each of size (d_model, d_model), for QKV and Ouptut projection O.
    # It takes input of shape (context_length, d_model) and produces output of shape (context_length, d_model)
    attention = (2 * config.context_length * (config.d_model ** 2)) * 4
    # Feed forward / SwigLU has 3 linear layers / matrix multiplication, each of size (d_model, d_ff)
    # It takes input of shape (context_length, d_model) and produces output of shape (context_length, d_model)
    swiglu = (2 * config.context_length * config.d_model * config.d_ff) * 3
    # Output linear layer is of size (d_model, vocab_size)
    out_linear =  (2 * config.d_model * config.vocab_size)

    total = (attention + swiglu) * config.n_layers + out_linear
    print(f"Attention consumes {attention * config.n_layers / total:.2%} flops, Feed forward {swiglu * config.n_layers / total:.2%} flops, and out linear {out_linear / total :.2%} flops")
    return total

In [28]:
GPT_2_XL = ModelConfig(50257, 1024, 48, 1600, 25, 6400)
print(f"Total model params is {model_params(GPT_2_XL):_}, total flops is {flops(GPT_2_XL):_}")

Attention has 491_520_000 params, Feed forward 1_474_560_000 params, out linear 80_411_200 params, emb 80_411_200 params
Attention consumes 1_006_632_960_000 flops, Feed forward 3_019_898_880_000 flops, and out linear 160_822_400 flops
Total model params is 2_127_057_600, total flops is 4_026_692_662_400


# Summary
- GPT-2 XL using our model arch would have 2.1B params, 4TFLOPS. Using float32, it would require 8.4GB memory to load the model.

This is slighly higher than the 1.6B params and 3.2 TFLOPS from the original GPT-2 XL model as SwiGLU has 3 matrix multiplies as opposed to 2 in the tranditional GPT-2 MLP layer. 

- The total of flops scale linearly with context_length and quadratically with d_model.

- MLP always consumes most flops, for about *75%* of total flops in this case with `d_ff = d_model * 4`, or *67%* with `d_ff = d_model * 8/3`

In [35]:
models = {
    "GPT_2_S": ModelConfig(vocab_size=50257, context_length=1024, n_layers=12, d_model=768, n_heads=12, d_ff=4*768),
    "GPT_2_M": ModelConfig(vocab_size=50257, context_length=1024, n_layers=24, d_model=1024, n_heads=16, d_ff=4*1024),
    "GPT_2_L": ModelConfig(vocab_size=50257, context_length=1024, n_layers=36, d_model=1280, n_heads=20, d_ff=4*1280),
    "GPT_2_XL": ModelConfig(vocab_size=50257, context_length=1024, n_layers=48, d_model=1600, n_heads=25, d_ff=4*1600),
    "GPT_2_XL_extra_context_length": ModelConfig(vocab_size=50257, context_length=16384, n_layers=48, d_model=1600, n_heads=25, d_ff=4*1600),

}

for model, config in models.items():
    print(f"Analyzing flops for {model}")
    print(f"total flops is {flops(config):_}")
    print()
    
    

Analyzing flops for GPT_2_S
Attention consumes 33.32% flops, Feed forward 66.64% flops, and out linear 0.04% flops
total flops is 174_023_370_240.0

Analyzing flops for GPT_2_M
Attention consumes 33.33% flops, Feed forward 66.66% flops, and out linear 0.02% flops
total flops is 618_578_216_960.0

Analyzing flops for GPT_2_L
Attention consumes 33.33% flops, Feed forward 66.66% flops, and out linear 0.01% flops
total flops is 1_449_680_120_320.0

Analyzing flops for GPT_2_XL
Attention consumes 33.33% flops, Feed forward 66.66% flops, and out linear 0.01% flops
total flops is 3_020_059_702_400.0

Analyzing flops for GPT_2_XL_extra_context_length
Attention consumes 25.00% flops, Feed forward 75.00% flops, and out linear 0.00% flops
total flops is 64_424_670_262_400

